# Constrain piece-wise constant pulses to vary smoothly

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from paraqeet.quantity import Quantity
from paraqeet.signal.envelopes import GaussEnvelope
from paraqeet.signal.pwc_generator import PWCGenerator
from paraqeet.measurement.smoothness import Smoothness
from paraqeet.optimisation_map import OptimisationMap
from paraqeet.optimisers.scipy_optimiser_gradient import ScipyOptimiserGradient

In [ ]:
# in seconds (See Eickbusch et al https://arxiv.org/abs/2111.06414 S9 A)
delta_sampling = 33e-9
n_pwc = 40  # number of piecewise constants in the pulse
t_final = n_pwc * delta_sampling  # for now just set up for trying
tlist = np.linspace(0, t_final, n_pwc + 1)
eps_qubit = 2 * np.pi * 1.0  # initial amplitude of the qubit (in MHz)
eps_max_qubit = 5 * eps_qubit  # maximum amplitude of the resonator (in MHz)
tone_qubit = GaussEnvelope(
    amplitude=Quantity(eps_qubit * 1e6, -eps_max_qubit * 1e6, eps_max_qubit * 1e6),
    t_final=Quantity(t_final, 1 / 2 * t_final, 2 * t_final),
)
gen_qubit = PWCGenerator(envelopes=[tone_qubit], max_amplitude=eps_max_qubit * 1e6, tlist=tlist)

In [ ]:
ts = np.linspace(0, t_final, 1001)


def plot_pulse(ts: np.ndarray, tone: GaussEnvelope):
    """Function to plot the pulse."""
    plt.plot(ts / 1e-9, tone_qubit.compute_output(ts) / 1e6 / 2 * np.pi, label="Smooth curve")
    plt.plot(ts / 1e-9, np.real(gen_qubit.generate_signal(ts)) / 1e6 / 2 * np.pi, ls="--", label="Inphase")
    plt.plot(
        ts / 1e-9,
        np.imag(gen_qubit.generate_signal(ts)) / 1e6,
        ls="--",
        label="Quadrature",
    )

    plt.xlabel("Time [in ns]")
    plt.ylabel("Amplitude [in MHz]")
    plt.legend()
    plt.show()


plot_pulse(ts, tone_qubit)

In [ ]:
optmap = OptimisationMap()
optmap.add(gen_qubit, gen_qubit.get_parameters())
# We add dummy parameters to check if the gradient is computed
# correctly by padding zeros
# optmap.add(tone_qubit, tone_qubit.get_parameters())
optmap.register_params_with_optimisables()

In [ ]:
smoothness = Smoothness(pwc_generator=gen_qubit)

In [ ]:
pwc_params = gen_qubit.get_parameters()
print(pwc_params)

In [ ]:
gen_qubit.generate_signal_gradient(0.0)

In [ ]:
print(smoothness.measure_with_gradient())

In [ ]:
print(smoothness.measure_with_gradient()[1][39])

We can check that the gradient has the correct shape

In [ ]:
print(smoothness.measure_with_gradient()[1].shape)

In [ ]:
opt = ScipyOptimiserGradient(smoothness, optimisables=optmap)
max_iter = 100
opt.set_options({"maxiter": max_iter})

In [ ]:
opt.optimise()

In [ ]:
smoothness.measure()

In [ ]:
smoothness.measure_with_gradient()

In [ ]:
plot_pulse(ts, tone_qubit)